Simulator: Multi Gaussian Expansion (Group)
===========================================

This script simulates an example strong lens on the 'group' scale, where there is a single primary lens galaxy
and two smaller extra galaxies nearby, whose mass contributes significantly to the ray-tracing and is therefore
included in the strong lens model.

The MGE feature is a **modeling** choice, not a property of the data itself. The simulated dataset is therefore
identical to the standard group simulator -- the same galaxy light profiles, mass profiles, and source galaxy
are used. This simulator exists to ensure that the MGE feature scripts can auto-simulate their required dataset.

If the dataset already exists (e.g. because the standard group simulator has already been run), this script
will not re-simulate it.

__Contents__

**Dataset Paths:** Output path for the simulated dataset.
**Grid:** Define the 2d grid of (y,x) coordinates for the simulation.
**Galaxy Centres:** Define the centres of the main lens galaxies and extra galaxies.
**Over Sampling:** Set up the adaptive over-sampling grid.
**Main Lens Galaxies:** The main lens galaxy at the origin.
**Extra Galaxies:** The two extra companion galaxies.
**Source Galaxy:** The source galaxy whose lensed images we simulate.
**Ray Tracing:** Use all galaxies to setup a tracer and simulate.
**Output:** Output the simulated dataset and metadata.

In [ ]:

from autoconf import jax_wrapper  # Sets JAX environment before other imports

from autoconf import setup_notebook; setup_notebook()

import numpy as np
from pathlib import Path
import autolens as al
import autolens.plot as aplt

__Dataset Paths__

The dataset is output to `dataset/group/simple`, the same path used by the standard group simulator.

In [ ]:
dataset_type = "group"
dataset_name = "simple"

dataset_path = Path("dataset", dataset_type, dataset_name)

__Dataset Auto-Simulation__

If the dataset already exists, we skip simulation. This prevents overwriting data that may have been
generated by the standard group simulator.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):

    """
    __Grid__

    Define the 2d grid of (y,x) coordinates that the lens and source galaxy images are evaluated and therefore
    simulated on.
    """
    grid = al.Grid2D.uniform(
        shape_native=(250, 250),
        pixel_scales=0.1,
    )

    """
    __Galaxy Centres__

    Define the centres of the main lens galaxies and extra galaxies.
    """
    main_lens_centres = [(0.0, 0.0)]
    extra_galaxies_centres = [(3.5, 2.5), (-4.4, -5.0)]

    """
    __Over Sampling__

    An adaptive oversampling scheme is applied at the centre of every galaxy in the group.
    """
    over_sample_size = al.util.over_sample.over_sample_size_via_radial_bins_from(
        grid=grid,
        sub_size_list=[32, 8, 2],
        radial_list=[0.3, 0.6],
        centre_list=main_lens_centres + extra_galaxies_centres,
    )

    grid = grid.apply_over_sampling(over_sample_size=over_sample_size)

    """
    Simulate a simple Gaussian PSF for the image.
    """
    psf = al.Convolver.from_gaussian(
        shape_native=(11, 11), sigma=0.1, pixel_scales=grid.pixel_scales
    )

    """
    To simulate the `Imaging` dataset we first create a simulator.
    """
    simulator = al.SimulatorImaging(
        exposure_time=300.0,
        psf=psf,
        background_sky_level=0.1,
        add_poisson_noise_to_data=True,
    )

    """
    __Main Lens Galaxies__

    The main lens galaxy is at the origin (0.0, 0.0). It has a spherical Sersic light profile and an isothermal
    mass profile.
    """
    lens_0 = al.Galaxy(
        redshift=0.5,
        bulge=al.lp.SersicSph(
            centre=(0.0, 0.0), intensity=0.7, effective_radius=2.0, sersic_index=4.0
        ),
        mass=al.mp.IsothermalSph(centre=(0.0, 0.0), einstein_radius=4.0),
    )

    main_lens_galaxies = [lens_0]

    """
    __Extra Galaxies__

    The two extra galaxies are companion galaxies near the lens system.
    """
    extra_galaxy_0 = al.Galaxy(
        redshift=0.5,
        bulge=al.lp.SersicSph(
            centre=(3.5, 2.5), intensity=0.9, effective_radius=0.8, sersic_index=3.0
        ),
        mass=al.mp.IsothermalSph(centre=(3.5, 2.5), einstein_radius=0.8),
    )

    extra_galaxy_1 = al.Galaxy(
        redshift=0.5,
        bulge=al.lp.SersicSph(
            centre=(-4.4, -5.0), intensity=0.9, effective_radius=0.8, sersic_index=3.0
        ),
        mass=al.mp.IsothermalSph(centre=(-4.4, -5.0), einstein_radius=1.0),
    )

    extra_galaxies = [extra_galaxy_0, extra_galaxy_1]

    """
    __Source Galaxy__

    The source galaxy uses a cored Sersic profile so that adaptive over-sampling is not required for the source.
    """
    source_galaxy = al.Galaxy(
        redshift=1.0,
        bulge=al.lp.SersicCore(
            centre=(0.0, 0.1),
            ell_comps=al.convert.ell_comps_from(axis_ratio=0.8, angle=60.0),
            intensity=3.0,
            effective_radius=0.4,
            sersic_index=1.0,
        ),
    )

    """
    __Ray Tracing__

    Use all galaxies to setup a tracer, which will generate the image for the simulated `Imaging` dataset.
    """
    tracer = al.Tracer(galaxies=main_lens_galaxies + extra_galaxies + [source_galaxy])

    aplt.plot_array(array=tracer.image_2d_from(grid=grid), title="Image")

    """
    __Dataset__

    Pass the simulator a tracer, which creates the image which is simulated as an imaging dataset.
    """
    dataset = simulator.via_tracer_from(tracer=tracer, grid=grid)

    aplt.subplot_imaging_dataset(dataset=dataset)

    """
    Output the simulated dataset to the dataset path as .fits files.
    """
    aplt.fits_imaging(
        dataset=dataset,
        data_path=dataset_path / "data.fits",
        psf_path=dataset_path / "psf.fits",
        noise_map_path=dataset_path / "noise_map.fits",
        overwrite=True,
    )

    """
    __Visualize__

    Output a subplot of the simulated dataset.
    """
    aplt.subplot_imaging_dataset(dataset=dataset)
    aplt.plot_array(array=dataset.data, title="Data")

    """
    __Tracer json__

    Save the `Tracer` in the dataset folder as a .json file.
    """
    al.output_to_json(
        obj=tracer,
        file_path=Path(dataset_path, "tracer.json"),
    )

    """
    __Centre JSON Files__

    Save the centres of the main lens galaxies and extra galaxies as JSON files.
    """
    al.output_to_json(
        obj=al.Grid2DIrregular(main_lens_centres),
        file_path=Path(dataset_path, "main_lens_centres.json"),
    )

    al.output_to_json(
        obj=al.Grid2DIrregular(extra_galaxies_centres),
        file_path=Path(dataset_path, "extra_galaxies_centres.json"),
    )

    """
    __Positions__

    Solve for the lensed positions of the source galaxy.
    """
    import os

    small_datasets = os.environ.pop("PYAUTO_SMALL_DATASETS", None)

    solver = al.PointSolver.for_grid(
        grid=al.Grid2D.uniform(shape_native=(500, 500), pixel_scales=0.1),
        pixel_scale_precision=0.001,
        magnification_threshold=0.01,
    )

    positions = solver.solve(
        tracer=tracer, source_plane_coordinate=source_galaxy.bulge.centre
    )

    if small_datasets is not None:
        os.environ["PYAUTO_SMALL_DATASETS"] = small_datasets

    al.output_to_json(
        obj=positions,
        file_path=dataset_path / "positions.json",
    )

Finished.